# Validate MERRA-2 null baseline

**Hypothesis:** `merra2_mean_annual_sw_kwh_m2` calls `.clip(aoi)` before returning the image.
The AOI is ~4.3 km². MERRA pixels are ~50-70 km. After clipping, no MERRA pixel *center* falls inside
the tiny clipped region, so `reduceRegion(mean, scale=50000)` returns `null` — even with `bestEffort`.

**Fix:** Remove `.clip(aoi)` from `merra2_mean_annual_sw_kwh_m2`. The image stays global;
`reduceRegion` with `geometry=aoi` already restricts the computation to the AOI bounds.

In [2]:
import os
import ee

PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "pv-mapping-india")
ee.Initialize(project=PROJECT_ID)

# Same AOI from the failing API response
coords = [
    [77.078098, 28.610869],
    [77.09809800000001, 28.610869],
    [77.09809800000001, 28.630869],
    [77.078098, 28.630869],
    [77.078098, 28.610869],
]
aoi = ee.Geometry.Polygon(coords)
print("AOI area km2:", aoi.area(1).divide(1e6).getInfo())

ModuleNotFoundError: No module named 'ee'

In [3]:
# --- Hunch test 1: does clip cause null? ---
COLLECTION = "NASA/GSFC/MERRA/rad/2"
BAND = "SWGDN"

col = ee.ImageCollection(COLLECTION).filterDate("2021-01-01", "2025-01-01").select(BAND)
img_global = col.sum().divide(1000.0).divide(4).rename("annual_SWGDN_kWh_m2")
img_clipped = img_global.clip(aoi)

SCALE = 50_000.0

raw_global = img_global.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=SCALE, maxPixels=1e9
).getInfo()

raw_clipped = img_clipped.reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=SCALE, maxPixels=1e9
).getInfo()

print("global (no clip):", raw_global)   # expect a real number
print("clipped to AOI: ", raw_clipped)   # expect null -> this is the bug

NameError: name 'ee' is not defined

In [ ]:
# --- Hunch test 2: centroid sample on global image ---
centroid = aoi.centroid(1)
sample = img_global.sample(region=centroid, scale=SCALE, numPixels=1, geometries=False).first().getInfo()
print("centroid sample:", sample["properties"] if sample else None)

In [ ]:
# --- Shadow model sanity check ---
# Expected: rooftop pixels should have LOW shadow fraction (not 97-100%).
# A building cannot shadow its own rooftop.
# We check by sampling shadow_retention at the centroid of a known building.

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

from scripts.penalties import shadow_retention_fraction
from scripts.datasets import get_open_buildings_temporal

buildings = get_open_buildings_temporal(aoi, year=2022)
height = buildings.select("building_height").setDefaultProjection(crs="EPSG:4326", scale=4)

retention = shadow_retention_fraction(height)

# Sample at the AOI centroid
centroid = aoi.centroid(1)
sample = retention.sample(region=centroid, scale=4, numPixels=1, geometries=False).first().getInfo()
print("shadow_retention at centroid:", sample["properties"] if sample else None)
# Expect: shadow_retention close to 1.0 (low shadow) for an open rooftop
# Bug was: always ~0.0-0.03 (97-100% shadow) because building shadowed itself